# WI-01: Weather Dataset Foundation

## 1. Objective

Establish a reliable, reproducible and weather-only data foundation
for BharatCrop Weather Intelligence 2.0.

## 2. Project Goal

Develop ML-powered weather intelligence for:
- Daily rainfall prediction (1–7 days ahead).
- Daily maximum temperature prediction (1–7 days ahead).
- Weather anomaly detection.
- Weather pattern discovery.
- Agricultural weather impact and risk analysis.

## 3. Historical Dataset

- Source: Existing validated NASA POWER / MERRA-2 weather archive.
- Historical period: 2013-01-01 to 2024-12-31.
- Spatial resolution: 2,520 weather grid cells.
- Temporal resolution: Daily.

## 4. Initial Weather Variables

- PRECTOTCORR: Precipitation.
- T2M_MAX: Maximum temperature.
- T2M_MIN: Minimum temperature.

## 5. Important Constraints

- No APY dependency in weather forecasting models.
- No random train-test splitting.
- No future information in predictive features.
- Preserve original datasets and artifacts.
- Validate all assumptions before model development.
- Historical predictions must not be presented as live forecasts.

## 6. Expected Output

A validated weather-only dataset and a reproducible
dataset profile for subsequent ML development.

In [2]:
# ============================================================
# WI-01 | CELL 2: IMPORTS AND CONFIGURATION
# ============================================================

from pathlib import Path
import json
import hashlib
import platform

import numpy as np
import pandas as pd

from IPython.display import display

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Expected historical period
START_DATE = "2013-01-01"
END_DATE = "2024-12-31"

# Weather variables selected for Weather Intelligence
WEATHER_VARIABLES = [
    "PRECTOTCORR",
    "T2M_MAX",
    "T2M_MIN"
]

print("Environment initialized successfully.")
print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Historical period:", START_DATE, "to", END_DATE)
print("Weather variables:", WEATHER_VARIABLES)

Environment initialized successfully.
Python: 3.13.5
Pandas: 2.3.3
NumPy: 2.5.4
Historical period: 2013-01-01 to 2024-12-31
Weather variables: ['PRECTOTCORR', 'T2M_MAX', 'T2M_MIN']


In [3]:
# ============================================================
# WI-01 | CELL 3: LOCATE PROJECT ROOT
# ============================================================

current_path = Path.cwd().resolve()

project_root = None

# Search current directory and parent directories
for candidate in [current_path, *current_path.parents]:

    weather_path = (
        candidate / "data" / "raw" / "weather_regional"
    )

    if weather_path.is_dir():
        project_root = candidate
        break

# Stop if the project directory cannot be identified
if project_root is None:
    raise FileNotFoundError(
        "BharatCrop project root not found.\n"
        "Expected directory: data/raw/weather_regional\n"
        "Check the notebook's working directory."
    )

# Define important paths
RAW_WEATHER_DIR = (
    project_root / "data" / "raw" / "weather_regional"
)

PROCESSED_DIR = (
    project_root / "data" / "processed"
)

print("Project root:", project_root)
print("Raw weather directory:", RAW_WEATHER_DIR)
print("Processed data directory:", PROCESSED_DIR)

print("\nDirectory checks:")
print("Raw weather exists:", RAW_WEATHER_DIR.exists())
print("Processed directory exists:", PROCESSED_DIR.exists())

Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Raw weather directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\raw\weather_regional
Processed data directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed

Directory checks:
Raw weather exists: True
Processed directory exists: True


In [4]:
# ============================================================
# WI-01 | CELL 4: PROCESSED DATA INVENTORY
# ============================================================

# Find all Parquet files inside the processed data directory
parquet_files = list(PROCESSED_DIR.rglob("*.parquet"))

inventory = []

for file_path in parquet_files:

    inventory.append({
        "file_name": file_path.name,
        "relative_path": str(file_path.relative_to(project_root)),
        "size_mb": round(
            file_path.stat().st_size / (1024 ** 2), 2
        )
    })

# Convert inventory into a DataFrame
parquet_inventory = pd.DataFrame(inventory)

if parquet_inventory.empty:

    print("No Parquet files found in the processed directory.")

else:

    # Display the available datasets
    parquet_inventory = parquet_inventory.sort_values(
        "relative_path"
    ).reset_index(drop=True)

    display(parquet_inventory)

print("\nTotal Parquet files:", len(parquet_files))

No Parquet files found in the processed directory.

Total Parquet files: 0


In [5]:
# ============================================================
# WI-01 | CELL 5: SEARCH EXISTING DATA FILES
# ============================================================

# File formats worth checking
supported_extensions = {
    ".parquet",
    ".pq",
    ".csv",
    ".feather",
    ".arrow",
    ".json",
    ".jsonl"
}

data_root = project_root / "data"

data_files = []

# Search inside the existing data directory
for file_path in data_root.rglob("*"):

    # Ignore directories
    if not file_path.is_file():
        continue

    # Keep only relevant data formats
    if file_path.suffix.lower() in supported_extensions:

        data_files.append({
            "file_name": file_path.name,
            "extension": file_path.suffix.lower(),
            "relative_path": str(
                file_path.relative_to(project_root)
            ),
            "size_mb": round(
                file_path.stat().st_size / (1024 ** 2), 2
            )
        })

# Display results
data_inventory = pd.DataFrame(data_files)

if data_inventory.empty:

    print("No supported data files found inside the data directory.")

else:

    data_inventory = data_inventory.sort_values(
        "relative_path"
    ).reset_index(drop=True)

    display(data_inventory)

print("\nTotal data files discovered:", len(data_files))

,file_name,extension,relative_path,size_mb
0,historical_food_grains_apy_2013-14_to_2024-25-...,.csv,data\processed\.ipynb_checkpoints\historical_f...,5.70
1,historical_food_grains_apy_2013-14_to_2024-25.csv,.csv,data\processed\historical_food_grains_apy_2013...,5.70
2,historical_food_grains_apy_cleaned_flagged.csv,.csv,data\processed\historical_food_grains_apy_clea...,15.20
3,historical_food_grains_apy_long_2013-14_to_202...,.csv,data\processed\historical_food_grains_apy_long...,41.53
4,nasa_power_weather_pilot_2013_2024.csv,.csv,data\processed\weather_pilot_2013_2024\nasa_po...,3.56
...,...,...,...,...
2060,R056_T2M_MIN_2020.json,.json,data\raw\weather_regional_pilot\R056_T2M_MIN_2...,0.16
2061,regional_PRECTOTCORR_R001_2013.json,.json,data\raw\weather_regional_test\regional_PRECTO...,0.36
2062,regional_PRECTOTCORR_R001_2020.json,.json,data\raw\weather_regional_test\regional_PRECTO...,0.36
2063,regional_T2M_MAX_R001_2013.json,.json,data\raw\weather_regional_test\regional_T2M_MA...,0.38



Total data files discovered: 2065


In [6]:
# ============================================================
# WI-01 | CELL 6: IDENTIFY WEATHER DATASET CANDIDATES
# ============================================================

# Search for weather-related files across the project
weather_candidates = []

for file_path in project_root.rglob("*"):

    if not file_path.is_file():
        continue

    # Ignore notebook checkpoint files
    if ".ipynb_checkpoints" in file_path.parts:
        continue

    # Consider only relevant processed data formats
    if file_path.suffix.lower() not in {
        ".parquet", ".pq", ".csv", ".feather", ".arrow"
    }:
        continue

    # Identify weather-related files by their path or filename
    searchable_text = (
        str(file_path.relative_to(project_root)).lower()
    )

    if "weather" in searchable_text or "nasa_power" in searchable_text:

        weather_candidates.append({
            "file_name": file_path.name,
            "extension": file_path.suffix.lower(),
            "relative_path": str(
                file_path.relative_to(project_root)
            ),
            "size_mb": round(
                file_path.stat().st_size / (1024 ** 2), 2
            )
        })

weather_inventory = pd.DataFrame(weather_candidates)

if weather_inventory.empty:

    print("No processed weather dataset candidates found.")

else:

    weather_inventory = weather_inventory.sort_values(
        ["extension", "relative_path"]
    ).reset_index(drop=True)

    display(weather_inventory)

print("\nTotal weather dataset candidates:", len(weather_candidates))
print(
    "Parquet candidates:",
    weather_inventory["extension"].isin([".parquet", ".pq"]).sum()
    if not weather_inventory.empty else 0
)

,file_name,extension,relative_path,size_mb
0,nasa_power_weather_pilot_2013_2024.csv,.csv,data\processed\weather_pilot_2013_2024\nasa_po...,3.56
1,apy_districts_without_weather_coverage.csv,.csv,data\processed\weather_regional\apy_districts_...,0.01
2,apy_weather_integrated_exploratory.csv,.csv,data\processed\weather_regional\apy_weather_in...,115.81
3,apy_weather_integration_summary.csv,.csv,data\processed\weather_regional\apy_weather_in...,0.00
4,apy_weather_proxy_district_year_coverage.csv,.csv,data\processed\weather_regional\apy_weather_pr...,0.00
5,crop_weather_integration_validation_summary.csv,.csv,data\processed\weather_regional\crop_weather_i...,0.00
6,district_weather_annual_2013_2024.csv,.csv,data\processed\weather_regional\district_weath...,2.01
7,district_weather_annual_cleaned_flagged_2013_2...,.csv,data\processed\weather_regional\district_weath...,2.23
8,district_weather_annual_preprocessed_2013_2024...,.csv,data\processed\weather_regional\district_weath...,1.04
9,nasa_power_grid_annual_summary_2013_2024.csv,.csv,data\processed\weather_regional\nasa_power_gri...,0.67



Total weather dataset candidates: 60
Parquet candidates: 12


In [7]:
# ============================================================
# WI-01 | CELL 7: INSPECT PARQUET PARTITIONS
# ============================================================

# Find all Parquet files in the project, excluding checkpoints
parquet_paths = [
    path
    for path in project_root.rglob("*.parquet")
    if ".ipynb_checkpoints" not in path.parts
]

# Sort paths for consistent output
parquet_paths = sorted(parquet_paths)

# Display exact file locations and sizes
partition_inventory = pd.DataFrame([
    {
        "file_name": path.name,
        "relative_path": str(path.relative_to(project_root)),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2)
    }
    for path in parquet_paths
])

display(partition_inventory)

# Calculate total dataset size
total_size_mb = sum(
    path.stat().st_size for path in parquet_paths
) / (1024 ** 2)

print("\nTotal Parquet partitions:", len(parquet_paths))
print("Combined size (MiB):", round(total_size_mb, 2))

# Identify the common parent directory
if parquet_paths:
    common_parent = Path(
        __import__("os").path.commonpath(
            [str(path.parent) for path in parquet_paths]
        )
    )

    print("Common dataset directory:", common_parent)

,file_name,relative_path,size_mb
0,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.21
1,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.18
2,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.24
3,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.18
4,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.23
5,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.20
6,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.29
7,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.33
8,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.29
9,part-00000.parquet,phase3b_weather_intelligence\data\processed\we...,5.28



Total Parquet partitions: 12
Combined size (MiB): 62.9
Common dataset directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_daily_v1


In [10]:
# ============================================================
# WI-01 | CELL 8: PARQUET SCHEMA AND METADATA
# ============================================================

# Import PyArrow for efficient Parquet inspection
import pyarrow as pa
import pyarrow.parquet as pq

# Define the canonical dataset directory
WEATHER_DATA_DIR = common_parent

# Collect metadata from every partition
partition_metadata = []

for file_path in parquet_paths:

    parquet_file = pq.ParquetFile(file_path)
    metadata = parquet_file.metadata

    partition_metadata.append({
        "file_name": file_path.name,
        "rows": metadata.num_rows,
        "row_groups": metadata.num_row_groups,
        "columns": metadata.num_columns,
        "size_mb": round(
            file_path.stat().st_size / (1024 ** 2), 2
        )
    })

# Display partition-level metadata
partition_metadata_df = pd.DataFrame(partition_metadata)

display(partition_metadata_df)

# Inspect schema from the first partition
first_parquet = pq.ParquetFile(parquet_paths[0])

print("\nDataset directory:", WEATHER_DATA_DIR)
print("\nParquet schema:")
print(first_parquet.schema)

print("\nArrow schema:")
print(first_parquet.schema_arrow)

# Calculate total rows across all partitions
total_rows = partition_metadata_df["rows"].sum()

print("\nTotal rows across partitions:", f"{total_rows:,}")
print("PyArrow version:", pa.__version__)

,file_name,rows,row_groups,columns,size_mb
0,part-00000.parquet,919800,4,22,5.21
1,part-00000.parquet,919800,4,22,5.18
2,part-00000.parquet,919800,4,22,5.24
3,part-00000.parquet,922320,4,22,5.18
4,part-00000.parquet,919800,4,22,5.23
5,part-00000.parquet,919800,4,22,5.20
6,part-00000.parquet,919800,4,22,5.29
7,part-00000.parquet,922320,4,22,5.33
8,part-00000.parquet,919800,4,22,5.29
9,part-00000.parquet,919800,4,22,5.28



Dataset directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_daily_v1

Parquet schema:
required group field_id=-1 schema {
  required binary field_id=-1 cell_id (String);
  required double field_id=-1 latitude;
  required double field_id=-1 longitude;
  required int32 field_id=-1 date (Date);
  required double field_id=-1 T2M_MAX;
  required double field_id=-1 T2M_MIN;
  required double field_id=-1 PRECTOTCORR;
  required binary field_id=-1 T2M_MAX_source_region_id (String);
  required binary field_id=-1 T2M_MAX_source_tags (String);
  required binary field_id=-1 T2M_MAX_unit (String);
  required binary field_id=-1 T2M_MIN_source_region_id (String);
  required binary field_id=-1 T2M_MIN_source_tags (String);
  required binary field_id=-1 T2M_MIN_unit (String);
  required binary field_id=-1 PRECTOTCORR_source_region_id (String);
  required binary field_id=-1 PRECTOTCORR_source_tags (String);
  required binary field_id=-1

In [11]:
# ============================================================
# WI-01 | CELL 9: PARTITION AND SCHEMA CONSISTENCY
# ============================================================

# Store the reference schema from the first partition
reference_schema = pq.ParquetFile(
    parquet_paths[0]
).schema_arrow

schema_results = []
partition_directories = []

# Inspect every Parquet partition
for file_path in parquet_paths:

    parquet_file = pq.ParquetFile(file_path)

    # Compare its schema with the reference schema
    schema_matches = (
        parquet_file.schema_arrow.equals(reference_schema)
    )

    # Identify the directory containing this partition
    relative_directory = str(
        file_path.parent.relative_to(WEATHER_DATA_DIR)
    )

    partition_directories.append(relative_directory)

    schema_results.append({
        "partition_directory": relative_directory,
        "rows": parquet_file.metadata.num_rows,
        "columns": parquet_file.metadata.num_columns,
        "schema_matches": schema_matches
    })

# Create a validation report
schema_validation_df = pd.DataFrame(schema_results)

display(schema_validation_df)

# Summarize the results
print("\nPartition count:", len(schema_validation_df))
print(
    "Unique partition directories:",
    schema_validation_df["partition_directory"].nunique()
)
print(
    "All schemas identical:",
    schema_validation_df["schema_matches"].all()
)
print(
    "All partitions contain 22 columns:",
    (schema_validation_df["columns"] == 22).all()
)
print(
    "Total rows:",
    f"{schema_validation_df['rows'].sum():,}"
)

# Check whether partition directories are unique
print(
    "Every partition has a unique directory:",
    len(set(partition_directories)) == len(partition_directories)
)

,partition_directory,rows,columns,schema_matches
0,year=2013,919800,22,True
1,year=2014,919800,22,True
2,year=2015,919800,22,True
3,year=2016,922320,22,True
4,year=2017,919800,22,True
5,year=2018,919800,22,True
6,year=2019,919800,22,True
7,year=2020,922320,22,True
8,year=2021,919800,22,True
9,year=2022,919800,22,True



Partition count: 12
Unique partition directories: 12
All schemas identical: True
All partitions contain 22 columns: True
Total rows: 11,045,160
Every partition has a unique directory: True


In [13]:
# ============================================================
# WI-01 | CELL 10: DATE AND GRID-CELL INTEGRITY
# ============================================================

import pyarrow.dataset as ds

# Explicitly use only the verified Parquet partitions
weather_dataset = ds.dataset(
    [str(path) for path in parquet_paths],
    format="parquet",
    partitioning="hive",
    partition_base_dir=str(WEATHER_DATA_DIR)
)

# Read only the columns required for validation
validation_table = weather_dataset.to_table(
    columns=[
        "cell_id",
        "latitude",
        "longitude",
        "date"
    ]
)

# Convert the validation data into Pandas
validation_df = validation_table.to_pandas()

validation_df["date"] = pd.to_datetime(
    validation_df["date"]
)

# ------------------------------------------------------------
# 1. Date coverage
# ------------------------------------------------------------

print("DATE COVERAGE")

print("Minimum date:", validation_df["date"].min())
print("Maximum date:", validation_df["date"].max())

# ------------------------------------------------------------
# 2. Grid-cell counts
# ------------------------------------------------------------

unique_cells = validation_df["cell_id"].nunique()

unique_coordinates = validation_df[
    ["latitude", "longitude"]
].drop_duplicates().shape[0]

print("\nGRID-CELL INFORMATION")
print("Unique cell IDs:", unique_cells)
print("Unique coordinate pairs:", unique_coordinates)

# ------------------------------------------------------------
# 3. Coordinate ranges
# ------------------------------------------------------------

print("\nCOORDINATE RANGES")

print(
    "Latitude:",
    validation_df["latitude"].min(),
    "to",
    validation_df["latitude"].max()
)

print(
    "Longitude:",
    validation_df["longitude"].min(),
    "to",
    validation_df["longitude"].max()
)

# ------------------------------------------------------------
# 4. Duplicate canonical keys
# ------------------------------------------------------------

duplicate_keys = validation_df.duplicated(
    subset=["cell_id", "date"]
).sum()

print("\nDUPLICATE CHECK")
print("Duplicate (cell_id, date) records:", duplicate_keys)

# ------------------------------------------------------------
# 5. Validation summary
# ------------------------------------------------------------

print("\nVALIDATION SUMMARY")

print(
    "Date range correct:",
    validation_df["date"].min() == pd.Timestamp(START_DATE)
    and validation_df["date"].max() == pd.Timestamp(END_DATE)
)

print("Expected grid cells found:", unique_cells == 2520)

print(
    "Cell IDs and coordinates consistent:",
    unique_cells == unique_coordinates
)

print("No duplicate canonical keys:", duplicate_keys == 0)

DATE COVERAGE
Minimum date: 2013-01-01 00:00:00
Maximum date: 2024-12-31 00:00:00

GRID-CELL INFORMATION
Unique cell IDs: 2520
Unique coordinate pairs: 2520

COORDINATE RANGES
Latitude: 7.0 to 34.5
Longitude: 69.375 to 96.875

DUPLICATE CHECK
Duplicate (cell_id, date) records: 0

VALIDATION SUMMARY
Date range correct: True
Expected grid cells found: True
Cell IDs and coordinates consistent: True
No duplicate canonical keys: True


In [14]:
# ============================================================
# WI-01 | CELL 11: WEATHER VALUE AND QUALITY VALIDATION
# ============================================================

# Read only weather measurements and QC columns
qc_columns = [
    "T2M_MAX",
    "T2M_MIN",
    "PRECTOTCORR",
    "qc_missing",
    "qc_sentinel",
    "qc_non_finite",
    "qc_negative_precipitation",
    "qc_tmin_gt_tmax",
    "qc_pass"
]

qc_table = weather_dataset.to_table(
    columns=qc_columns
)

qc_df = qc_table.to_pandas()

# ------------------------------------------------------------
# 1. Missing values
# ------------------------------------------------------------

print("MISSING VALUES")

display(qc_df[
    ["T2M_MAX", "T2M_MIN", "PRECTOTCORR"]
].isna().sum().to_frame("missing_count"))

# ------------------------------------------------------------
# 2. Non-finite values
# ------------------------------------------------------------

print("\nNON-FINITE VALUES")

for column in WEATHER_VARIABLES:

    non_finite_count = (
        ~np.isfinite(qc_df[column])
    ).sum()

    print(column, ":", non_finite_count)

# ------------------------------------------------------------
# 3. Physical consistency checks
# ------------------------------------------------------------

negative_precipitation = (
    qc_df["PRECTOTCORR"] < 0
).sum()

invalid_temperature_order = (
    qc_df["T2M_MIN"] > qc_df["T2M_MAX"]
).sum()

print("\nPHYSICAL CONSISTENCY")
print("Negative precipitation records:", negative_precipitation)
print("Tmin greater than Tmax records:", invalid_temperature_order)

# ------------------------------------------------------------
# 4. Stored quality-control flags
# ------------------------------------------------------------

qc_flag_columns = [
    "qc_missing",
    "qc_sentinel",
    "qc_non_finite",
    "qc_negative_precipitation",
    "qc_tmin_gt_tmax"
]

print("\nQUALITY-CONTROL FLAG COUNTS")

display(
    qc_df[qc_flag_columns]
    .sum()
    .astype("int64")
    .to_frame("flagged_records")
)

# ------------------------------------------------------------
# 5. Overall QC status
# ------------------------------------------------------------

print("\nOVERALL QUALITY CONTROL")

print("Records passing QC:", int(qc_df["qc_pass"].sum()))
print("Records failing QC:", int((~qc_df["qc_pass"]).sum()))

print(
    "QC pass percentage:",
    round(qc_df["qc_pass"].mean() * 100, 4),
    "%"
)

MISSING VALUES


,missing_count
T2M_MAX,0
T2M_MIN,0
PRECTOTCORR,0



NON-FINITE VALUES
PRECTOTCORR : 0
T2M_MAX : 0
T2M_MIN : 0

PHYSICAL CONSISTENCY
Negative precipitation records: 0
Tmin greater than Tmax records: 0

QUALITY-CONTROL FLAG COUNTS


,flagged_records
qc_missing,0
qc_sentinel,0
qc_non_finite,0
qc_negative_precipitation,0
qc_tmin_gt_tmax,0



OVERALL QUALITY CONTROL
Records passing QC: 11045160
Records failing QC: 0
QC pass percentage: 100.0 %


In [15]:
# ============================================================
# WI-01 | CELL 12: SOURCE AND UNIT VALIDATION
# ============================================================

# Read only provenance columns
provenance_columns = [
    "T2M_MAX_source_region_id",
    "T2M_MAX_source_tags",
    "T2M_MAX_unit",
    "T2M_MIN_source_region_id",
    "T2M_MIN_source_tags",
    "T2M_MIN_unit",
    "PRECTOTCORR_source_region_id",
    "PRECTOTCORR_source_tags",
    "PRECTOTCORR_unit"
]

provenance_table = weather_dataset.to_table(
    columns=provenance_columns
)

provenance_df = provenance_table.to_pandas()

# ------------------------------------------------------------
# 1. Inspect units
# ------------------------------------------------------------

unit_columns = [
    "T2M_MAX_unit",
    "T2M_MIN_unit",
    "PRECTOTCORR_unit"
]

print("UNIQUE MEASUREMENT UNITS")

for column in unit_columns:

    print(f"\n{column}")
    print(provenance_df[column].value_counts(dropna=False))

# ------------------------------------------------------------
# 2. Inspect source tags
# ------------------------------------------------------------

source_tag_columns = [
    "T2M_MAX_source_tags",
    "T2M_MIN_source_tags",
    "PRECTOTCORR_source_tags"
]

print("\nSOURCE TAGS")

for column in source_tag_columns:

    print(f"\n{column}")
    print(provenance_df[column].value_counts(dropna=False))

# ------------------------------------------------------------
# 3. Inspect source region identifiers
# ------------------------------------------------------------

region_columns = [
    "T2M_MAX_source_region_id",
    "T2M_MIN_source_region_id",
    "PRECTOTCORR_source_region_id"
]

print("\nSOURCE REGION IDENTIFIERS")

for column in region_columns:

    print(
        column,
        "| Unique values:",
        provenance_df[column].nunique(),
        "| Missing values:",
        provenance_df[column].isna().sum()
    )

# ------------------------------------------------------------
# 4. Check expected units
# ------------------------------------------------------------

expected_units = {
    "T2M_MAX_unit": "C",
    "T2M_MIN_unit": "C",
    "PRECTOTCORR_unit": "mm/day"
}

print("\nEXPECTED UNIT VALIDATION")

for column, expected_unit in expected_units.items():

    observed_units = set(
        provenance_df[column].dropna().unique()
    )

    print(
        column,
        "| Expected:",
        expected_unit,
        "| Observed:",
        observed_units,
        "| Passed:",
        observed_units == {expected_unit}
    )

UNIQUE MEASUREMENT UNITS

T2M_MAX_unit
T2M_MAX_unit
C    11045160
Name: count, dtype: int64

T2M_MIN_unit
T2M_MIN_unit
C    11045160
Name: count, dtype: int64

PRECTOTCORR_unit
PRECTOTCORR_unit
mm/day    11045160
Name: count, dtype: int64

SOURCE TAGS

T2M_MAX_source_tags
T2M_MAX_source_tags
["MERRA2","POWER"]    11045160
Name: count, dtype: int64

T2M_MIN_source_tags
T2M_MIN_source_tags
["MERRA2","POWER"]    11045160
Name: count, dtype: int64

PRECTOTCORR_source_tags
PRECTOTCORR_source_tags
["MERRA2"]    11045160
Name: count, dtype: int64

SOURCE REGION IDENTIFIERS
T2M_MAX_source_region_id | Unique values: 56 | Missing values: 0
T2M_MIN_source_region_id | Unique values: 56 | Missing values: 0
PRECTOTCORR_source_region_id | Unique values: 56 | Missing values: 0

EXPECTED UNIT VALIDATION
T2M_MAX_unit | Expected: C | Observed: {'C'} | Passed: True
T2M_MIN_unit | Expected: C | Observed: {'C'} | Passed: True
PRECTOTCORR_unit | Expected: mm/day | Observed: {'mm/day'} | Passed: True


In [16]:
# ============================================================
# WI-01 | CELL 13: WEATHER VARIABLE DISTRIBUTION PROFILE
# ============================================================

# Read only the three weather variables
weather_values_table = weather_dataset.to_table(
    columns=WEATHER_VARIABLES
)

weather_values_df = weather_values_table.to_pandas()

# Calculate descriptive statistics
weather_profile = weather_values_df.describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

# Add additional information
weather_profile["missing_count"] = (
    weather_values_df.isna().sum()
)

weather_profile["unique_values"] = (
    weather_values_df.nunique()
)

# Display the profile
display(weather_profile.round(3))

# ------------------------------------------------------------
# 1. Rainfall-specific profile
# ------------------------------------------------------------

rainfall = weather_values_df["PRECTOTCORR"]

print("\nRAINFALL DISTRIBUTION")

print("Zero-rainfall records:", int((rainfall == 0).sum()))

print(
    "Positive-rainfall records:",
    int((rainfall > 0).sum())
)

print(
    "Positive-rainfall percentage:",
    round((rainfall > 0).mean() * 100, 2),
    "%"
)

# ------------------------------------------------------------
# 2. Temperature consistency
# ------------------------------------------------------------

temperature_range = (
    weather_values_df["T2M_MAX"]
    - weather_values_df["T2M_MIN"]
)

print("\nTEMPERATURE RANGE")

print("Minimum daily temperature range:", round(temperature_range.min(), 3))
print("Maximum daily temperature range:", round(temperature_range.max(), 3))
print("Average daily temperature range:", round(temperature_range.mean(), 3))

# ------------------------------------------------------------
# 3. Final dataset dimensions
# ------------------------------------------------------------

print("\nDATASET DIMENSIONS")

print("Total observations:", f"{len(weather_values_df):,}")
print("Weather variables:", len(WEATHER_VARIABLES))
print("Total missing values:", int(weather_values_df.isna().sum().sum()))

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_count,unique_values
PRECTOTCORR,11045160.0,3.690,9.537,0.00,0.00,0.00,0.00,0.27,3.41,17.79,41.26,1038.34,0,17027
T2M_MAX,11045160.0,25.928,10.951,-24.19,-9.39,-0.36,25.59,28.57,30.76,39.75,43.87,51.51,0,7247
T2M_MIN,11045160.0,17.936,13.542,-42.81,-23.51,-13.77,12.93,23.97,27.51,29.06,30.12,37.98,0,7513



RAINFALL DISTRIBUTION
Zero-rainfall records: 3831188
Positive-rainfall records: 7213972
Positive-rainfall percentage: 65.31 %

TEMPERATURE RANGE
Minimum daily temperature range: 0.04
Maximum daily temperature range: 30.2
Average daily temperature range: 7.993

DATASET DIMENSIONS
Total observations: 11,045,160
Weather variables: 3
Total missing values: 0


In [17]:
# ============================================================
# WI-01 | CELL 14: EXTREME WEATHER OBSERVATION REVIEW
# ============================================================

# Read weather values with their geographic and temporal context
extreme_columns = [
    "cell_id",
    "latitude",
    "longitude",
    "date",
    "PRECTOTCORR",
    "T2M_MAX",
    "T2M_MIN"
]

extreme_table = weather_dataset.to_table(
    columns=extreme_columns
)

extreme_df = extreme_table.to_pandas()

# ------------------------------------------------------------
# 1. Highest rainfall observations
# ------------------------------------------------------------

print("TOP 20 HIGHEST RAINFALL OBSERVATIONS")

top_rainfall = extreme_df.nlargest(
    20, "PRECTOTCORR"
)[
    [
        "cell_id",
        "latitude",
        "longitude",
        "date",
        "PRECTOTCORR"
    ]
]

display(top_rainfall)

# ------------------------------------------------------------
# 2. Highest maximum temperatures
# ------------------------------------------------------------

print("\nTOP 10 HIGHEST MAXIMUM TEMPERATURES")

top_tmax = extreme_df.nlargest(
    10, "T2M_MAX"
)[
    [
        "cell_id",
        "latitude",
        "longitude",
        "date",
        "T2M_MAX"
    ]
]

display(top_tmax)

# ------------------------------------------------------------
# 3. Lowest minimum temperatures
# ------------------------------------------------------------

print("\nTOP 10 LOWEST MINIMUM TEMPERATURES")

lowest_tmin = extreme_df.nsmallest(
    10, "T2M_MIN"
)[
    [
        "cell_id",
        "latitude",
        "longitude",
        "date",
        "T2M_MIN"
    ]
]

display(lowest_tmin)

# ------------------------------------------------------------
# 4. Count observations above investigation thresholds
# ------------------------------------------------------------

print("\nEXTREME VALUE COUNTS")

print(
    "Rainfall above 100 mm/day:",
    int((extreme_df["PRECTOTCORR"] > 100).sum())
)

print(
    "Rainfall above 200 mm/day:",
    int((extreme_df["PRECTOTCORR"] > 200).sum())
)

print(
    "Maximum temperature above 45°C:",
    int((extreme_df["T2M_MAX"] > 45).sum())
)

print(
    "Minimum temperature below -20°C:",
    int((extreme_df["T2M_MIN"] < -20).sum())
)

TOP 20 HIGHEST RAINFALL OBSERVATIONS


,cell_id,latitude,longitude,date,PRECTOTCORR
4138172,cell_1250,20.5,91.250,2017-07-22,1038.34
2133797,cell_0806,15.5,95.000,2015-01-08,920.81
4138902,cell_1252,20.5,92.500,2017-07-22,856.07
4155327,cell_1297,21.0,92.500,2017-07-22,753.16
4138537,cell_1251,20.5,91.875,2017-07-22,743.72
5962493,cell_1208,20.0,93.125,2019-09-11,720.73
3217828,cell_1252,20.5,92.500,2016-07-15,672.45
4154962,cell_1296,21.0,91.875,2017-07-22,670.96
6948175,cell_1385,22.0,91.250,2020-05-25,666.37
5107328,cell_1385,22.0,91.250,2018-10-11,662.43



TOP 10 HIGHEST MAXIMUM TEMPERATURES


,cell_id,latitude,longitude,date,T2M_MAX
10814727,cell_1890,28.0,69.375,2024-05-27,51.51
10831563,cell_1936,28.5,70.000,2024-05-27,51.30
10848400,cell_1982,29.0,70.625,2024-05-28,51.30
10831197,cell_1935,28.5,69.375,2024-05-27,51.18
10831564,cell_1936,28.5,70.000,2024-05-28,51.18
10748848,cell_1710,26.0,69.375,2024-05-28,51.17
10831930,cell_1937,28.5,70.625,2024-05-28,51.11
10831562,cell_1936,28.5,70.000,2024-05-26,51.09
10815092,cell_1891,28.0,70.000,2024-05-26,51.05
10848399,cell_1982,29.0,70.625,2024-05-27,51.00



TOP 10 LOWEST MINIMUM TEMPERATURES


,cell_id,latitude,longitude,date,T2M_MIN
763238,cell_2091,30.0,82.500,2013-01-24,-42.81
6430164,cell_2489,34.5,78.125,2019-12-26,-42.19
6396616,cell_2398,33.5,77.500,2019-01-27,-41.90
6414104,cell_2445,34.0,78.750,2019-12-26,-41.20
6430163,cell_2489,34.5,78.125,2019-12-25,-40.75
7206811,cell_2092,30.0,83.125,2020-01-20,-40.35
6284885,cell_2091,30.0,82.500,2019-12-17,-40.31
7272691,cell_2272,32.0,83.125,2020-01-20,-40.20
6396615,cell_2398,33.5,77.500,2019-01-26,-40.17
6397670,cell_2400,33.5,78.750,2019-12-17,-40.10



EXTREME VALUE COUNTS
Rainfall above 100 mm/day: 11974
Rainfall above 200 mm/day: 1620
Maximum temperature above 45°C: 53554
Minimum temperature below -20°C: 254599


In [18]:
# ============================================================
# WI-01 | CELL 15: EXTREME RAINFALL INVESTIGATION
# ============================================================

# Extract dates associated with the 20 highest rainfall values
extreme_dates = (
    pd.to_datetime(top_rainfall["date"])
    .dt.date
    .unique()
    .tolist()
)

# Select only records above 200 mm/day on those dates
rainfall_filter = (
    ds.field("date").isin(extreme_dates)
    & (ds.field("PRECTOTCORR") > 200)
)

# Read relevant measurements and provenance information
extreme_rainfall_table = weather_dataset.to_table(
    filter=rainfall_filter,
    columns=[
        "cell_id",
        "latitude",
        "longitude",
        "date",
        "PRECTOTCORR",
        "PRECTOTCORR_source_region_id",
        "PRECTOTCORR_source_tags",
        "PRECTOTCORR_unit",
        "qc_pass",
        "qc_negative_precipitation",
        "qc_non_finite",
        "qc_sentinel"
    ]
)

extreme_rainfall_df = extreme_rainfall_table.to_pandas()

# ------------------------------------------------------------
# 1. Extreme rainfall observations
# ------------------------------------------------------------

print("EXTREME RAINFALL RECORDS ON TOP-20 EVENT DATES")

extreme_rainfall_df = extreme_rainfall_df.sort_values(
    "PRECTOTCORR",
    ascending=False
).reset_index(drop=True)

display(extreme_rainfall_df.head(50))

# ------------------------------------------------------------
# 2. Source-region distribution
# ------------------------------------------------------------

print("\nSOURCE-REGION DISTRIBUTION")

display(
    extreme_rainfall_df[
        "PRECTOTCORR_source_region_id"
    ].value_counts().to_frame("record_count")
)

# ------------------------------------------------------------
# 3. Extreme rainfall by date
# ------------------------------------------------------------

print("\nEXTREME RAINFALL BY DATE")

daily_extreme_summary = (
    extreme_rainfall_df
    .groupby("date")
    .agg(
        extreme_grid_cells=("cell_id", "nunique"),
        maximum_rainfall=("PRECTOTCORR", "max"),
        mean_extreme_rainfall=("PRECTOTCORR", "mean")
    )
    .sort_values("maximum_rainfall", ascending=False)
)

display(daily_extreme_summary.round(2))

# ------------------------------------------------------------
# 4. QC and provenance summary
# ------------------------------------------------------------

print("\nSOURCE AND QC SUMMARY")

print(
    "Unique source tags:",
    extreme_rainfall_df[
        "PRECTOTCORR_source_tags"
    ].unique()
)

print(
    "Unique units:",
    extreme_rainfall_df[
        "PRECTOTCORR_unit"
    ].unique()
)

print(
    "Records passing QC:",
    int(extreme_rainfall_df["qc_pass"].sum())
)

print(
    "Records failing QC:",
    int((~extreme_rainfall_df["qc_pass"]).sum())
)

EXTREME RAINFALL RECORDS ON TOP-20 EVENT DATES


,cell_id,latitude,longitude,date,PRECTOTCORR,PRECTOTCORR_source_region_id,PRECTOTCORR_source_tags,PRECTOTCORR_unit,qc_pass,qc_negative_precipitation,qc_non_finite,qc_sentinel
0,cell_1250,20.5,91.250,2017-07-22,1038.34,R027,"[""MERRA2""]",mm/day,True,False,False,False
1,cell_0806,15.5,95.000,2015-01-08,920.81,R021,"[""MERRA2""]",mm/day,True,False,False,False
2,cell_1252,20.5,92.500,2017-07-22,856.07,R027,"[""MERRA2""]",mm/day,True,False,False,False
3,cell_1297,21.0,92.500,2017-07-22,753.16,R027,"[""MERRA2""]",mm/day,True,False,False,False
4,cell_1251,20.5,91.875,2017-07-22,743.72,R027,"[""MERRA2""]",mm/day,True,False,False,False
5,cell_1208,20.0,93.125,2019-09-11,720.73,R027,"[""MERRA2""]",mm/day,True,False,False,False
6,cell_1252,20.5,92.500,2016-07-15,672.45,R027,"[""MERRA2""]",mm/day,True,False,False,False
7,cell_1296,21.0,91.875,2017-07-22,670.96,R027,"[""MERRA2""]",mm/day,True,False,False,False
8,cell_1385,22.0,91.250,2020-05-25,666.37,R027,"[""MERRA2""]",mm/day,True,False,False,False
9,cell_1385,22.0,91.250,2018-10-11,662.43,R027,"[""MERRA2""]",mm/day,True,False,False,False



SOURCE-REGION DISTRIBUTION


,record_count
PRECTOTCORR_source_region_id,
R027,107
R021,15
R020,13
R028,8



EXTREME RAINFALL BY DATE


,extreme_grid_cells,maximum_rainfall,mean_extreme_rainfall
date,,,
2017-07-22,17,1038.34,497.31
2015-01-08,5,920.81,440.07
2019-09-11,29,720.73,360.09
2016-07-15,20,672.45,413.14
2020-05-25,3,666.37,541.64
2018-10-11,18,662.43,352.35
2015-09-01,17,621.93,355.39
2016-07-03,34,617.02,344.35



SOURCE AND QC SUMMARY
Unique source tags: ['["MERRA2"]']
Unique units: ['mm/day']
Records passing QC: 143
Records failing QC: 0


In [19]:
# ============================================================
# WI-01 | CELL 16: CREATE DATASET PROFILE
# ============================================================

# ------------------------------------------------------------
# 1. Build the dataset profile
# ------------------------------------------------------------

dataset_profile = {
    "project": "BharatCrop Intelligence",
    "module": "Weather Intelligence 2.0",
    "notebook": "WI-01",
    "dataset_name": "weather_daily_v1",
    "dataset_version": "1.0.0",

    "source": "NASA POWER / MERRA-2",
    "start_date": START_DATE,
    "end_date": END_DATE,
    "date_semantics": "UTC calendar date",

    "total_observations": int(total_rows),
    "total_partitions": int(len(parquet_paths)),
    "total_grid_cells": int(unique_cells),

    "latitude_range": [
        float(validation_df["latitude"].min()),
        float(validation_df["latitude"].max())
    ],

    "longitude_range": [
        float(validation_df["longitude"].min()),
        float(validation_df["longitude"].max())
    ],

    "weather_variables": {
        "PRECTOTCORR": {
            "description": "Daily precipitation",
            "unit": "mm/day"
        },
        "T2M_MAX": {
            "description": "Daily maximum temperature",
            "unit": "C"
        },
        "T2M_MIN": {
            "description": "Daily minimum temperature",
            "unit": "C"
        }
    },

    "integrity_validation": {
        "schemas_consistent": bool(
            schema_validation_df["schema_matches"].all()
        ),
        "date_range_valid": True,
        "unique_cells": int(unique_cells),
        "unique_coordinate_pairs": int(unique_coordinates),
        "duplicate_cell_date_records": int(duplicate_keys),
        "missing_weather_values": int(
            qc_df[WEATHER_VARIABLES].isna().sum().sum()
        ),
        "non_finite_weather_values": int(
            sum(
                (~np.isfinite(qc_df[column])).sum()
                for column in WEATHER_VARIABLES
            )
        ),
        "negative_precipitation_records": int(
            (qc_df["PRECTOTCORR"] < 0).sum()
        ),
        "tmin_greater_than_tmax_records": int(
            (qc_df["T2M_MIN"] > qc_df["T2M_MAX"]).sum()
        ),
        "qc_pass_percentage": round(
            float(qc_df["qc_pass"].mean() * 100), 4
        )
    },

    "extreme_value_investigation": {
        "rainfall_above_200_mm_on_top_20_event_dates": int(
            len(extreme_rainfall_df)
        ),
        "extreme_rainfall_requires_further_validation": True,
        "automatic_outlier_removal": False
    },

    "limitations": [
        "Grid cells are not administrative districts.",
        "District-level aggregation requires validated spatial mapping.",
        "Historical data ends on 2024-12-31.",
        "Historical data alone cannot provide live 2026 forecasts.",
        "Extreme rainfall observations require further investigation."
    ],

    "forecasting_constraints": [
        "Weather-only forecasting; no APY dependency.",
        "Forecast horizons: 1 to 7 days.",
        "Use chronological evaluation, not random splitting.",
        "Prevent future information leakage.",
        "Compare ML models against forecasting baselines."
    ]
}

# ------------------------------------------------------------
# 2. Create a versioned output location
# ------------------------------------------------------------

PROFILE_DIR = (
    project_root
    / "phase3b_weather_intelligence"
    / "reports"
    / "weather_intelligence_v2"
)

PROFILE_DIR.mkdir(parents=True, exist_ok=True)

# Never overwrite an existing profile
profile_path = PROFILE_DIR / "WI_01_dataset_profile.json"

version = 2

while profile_path.exists():

    profile_path = PROFILE_DIR / (
        f"WI_01_dataset_profile_v{version}.json"
    )

    version += 1

# ------------------------------------------------------------
# 3. Save the profile
# ------------------------------------------------------------

with open(profile_path, "w", encoding="utf-8") as file:

    json.dump(
        dataset_profile,
        file,
        indent=4
    )

print("WI-01 dataset profile created successfully.")
print("Profile location:", profile_path)

# ------------------------------------------------------------
# 4. Display final status
# ------------------------------------------------------------

print("\nFINAL DATASET PROFILE")

print("Observations:", f"{dataset_profile['total_observations']:,}")
print("Grid cells:", dataset_profile["total_grid_cells"])
print("Partitions:", dataset_profile["total_partitions"])
print("Historical coverage:", START_DATE, "to", END_DATE)

print(
    "QC pass percentage:",
    dataset_profile["integrity_validation"]["qc_pass_percentage"],
    "%"
)

print(
    "Extreme rainfall investigation pending:",
    dataset_profile["extreme_value_investigation"][
        "extreme_rainfall_requires_further_validation"
    ]
)

WI-01 dataset profile created successfully.
Profile location: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_01_dataset_profile.json

FINAL DATASET PROFILE
Observations: 11,045,160
Grid cells: 2520
Partitions: 12
Historical coverage: 2013-01-01 to 2024-12-31
QC pass percentage: 100.0 %
Extreme rainfall investigation pending: True


# WI-01: Weather Dataset Foundation — Completion Summary

## 1. Objective Achieved

Established a reproducible and validated weather-only dataset foundation for BharatCrop Intelligence — Weather Intelligence 2.0.

The existing historical weather archive was located, inspected and validated without modifying or regenerating the original dataset.

## 2. Dataset Summary

| Property | Validated Result |
|---|---|
| Dataset | `weather_daily_v1` |
| Dataset version | 1.0.0 |
| Historical coverage | 2013-01-01 to 2024-12-31 |
| Total observations | 11,045,160 |
| Weather grid cells | 2,520 |
| Yearly partitions | 12 |
| Total columns | 22 |
| Temporal resolution | Daily |
| Date semantics | UTC calendar date |

### Weather Variables

- `PRECTOTCORR`: Daily precipitation (mm/day).
- `T2M_MAX`: Daily maximum temperature (°C).
- `T2M_MIN`: Daily minimum temperature (°C).

## 3. Validation Results

The following checks were completed:

- Verified all 12 Parquet partitions and schema consistency.
- Confirmed historical date coverage.
- Verified 2,520 unique grid cells and coordinate pairs.
- Confirmed zero duplicate `(cell_id, date)` records.
- Checked missing and non-finite weather values.
- Verified precipitation and temperature physical consistency checks.
- Validated measurement units and source provenance.
- Reviewed weather-variable distributions and extreme observations.
- Confirmed that all 11,045,160 records passed the stored quality-control checks.

**Overall stored QC pass rate: 100%.**

## 4. Important Findings

- Rainfall has a zero-heavy, highly skewed distribution.
- Extreme rainfall observations require additional investigation.
- Temperature distributions exhibit substantial geographic and seasonal variation.
- Extreme temperature observations require contextual interpretation.
- Weather grid cells must not be treated as administrative districts.

## 5. Known Limitations and Pending Work

1. Extreme rainfall observations have not yet been independently validated.
2. District-level spatial aggregation requires a validated geographic mapping.
3. The historical archive ends on 31 December 2024.
4. Historical data alone cannot support live operational forecasts for 2026.
5. Fresh observations or an operational forecast source will be required for live forecasting.

No observations were automatically deleted, clipped or modified during this notebook.

## 6. Reproducibility and Artifact

Dataset profile generated successfully:

`reports/weather_intelligence_v2/WI_01_dataset_profile.json`

The original Parquet dataset and existing project artifacts remain preserved.

## 7. Handoff to WI-02

The validated historical weather dataset is ready for exploratory analysis.

The next notebook, **WI-02: Exploratory Weather Intelligence**, will investigate:

- Seasonal and monthly rainfall behaviour.
- Monsoon and non-monsoon weather patterns.
- Geographic variation in rainfall and temperature.
- Long-term temperature and rainfall trends.
- Extreme-weather distributions and anomalies.
- Temporal and spatial relationships relevant to forecasting.

These findings will guide the design of leakage-safe forecasting features and subsequent machine-learning experiments.

## Final Status

**WI-01: WEATHER DATASET FOUNDATION — COMPLETED**

The historical weather dataset foundation is established. Outstanding data limitations are documented and will be addressed in the relevant downstream stages.

**Next Notebook:** WI-02 — Exploratory Weather Intelligence.